# Desafio extra: ANOVA e Tukey por formato

O enunciado pede ANOVA combinada ao teste de Tukey **caso a comparação dos intervalos de confiança não permita concluir**.

**Regra usada:** para cada formato, comparam-se os ICs de 95% dos três pares de algoritmos. Se algum par tiver ICs sobrepostos, o desafio extra é acionado para aquele formato. Se nenhum par se sobrepõe, a conclusão sai dos ICs.

**Análise em cada formato acionado:**
1. **Premissas:** normalidade (Shapiro-Wilk, por algoritmo) e homogeneidade das variâncias (Levene).
2. **ANOVA de um fator:** H0: as três médias são iguais. H1: pelo menos uma é diferente.
3. **Tukey HSD:** compara os três pares, com correção para múltiplas comparações.

Os resultados aparecem no próprio notebook. Nenhum arquivo é gravado.

## 0. Preparação

In [1]:
from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from scipy import stats

RAIZ = Path.cwd()
if not (RAIZ / "dados").exists():
    RAIZ = RAIZ.parent
DADOS = RAIZ / "dados"

FORMATOS = ["QR-QI-ME", "QR-ME-QI", "QI-QR-ME", "QI-ME-QR", "ME-QR-QI", "ME-QI-QR"]
ALGORITMOS = ["Quick Sort Recursivo", "Quick Sort Iterativo", "Merge Sort"]
SIGLA = {"Quick Sort Recursivo": "QR", "Quick Sort Iterativo": "QI", "Merge Sort": "ME"}
CONFIANCA = 0.95
ALFA = 1 - CONFIANCA

arquivos = [DADOS / "formato_{}.csv".format(f) for f in FORMATOS]
faltando = [a.name for a in arquivos if not a.exists()]
if faltando:
    raise FileNotFoundError("Arquivos nao encontrados: " + ", ".join(faltando))

df = pd.concat([pd.read_csv(a) for a in arquivos], ignore_index=True)
validas = df[df["descartada"] == "nao"].copy()


def resumo(x):
    x = np.asarray(x, dtype=float)
    n = len(x)
    media = x.mean()
    desvio = x.std(ddof=1)
    t = stats.t.ppf((1 + CONFIANCA) / 2, n - 1)
    margem = t * desvio / np.sqrt(n)
    return {"n": n, "media": media, "desvio": desvio, "margem": margem,
            "ic_inf": media - margem, "ic_sup": media + margem}


def sobrepoe(a, b):
    return a["ic_inf"] <= b["ic_sup"] and b["ic_inf"] <= a["ic_sup"]


print("Rodadas validas:", len(validas))

Rodadas validas: 852


## 1. Decisão: quais formatos precisam do desafio extra

In [2]:
def decide(formato):
    sub = validas[validas["formato"] == formato]
    grupos = {a: sub.loc[sub["algoritmo"] == a, "tempo_ms"].to_numpy(dtype=float)
              for a in ALGORITMOS}
    resumos = {a: resumo(g) for a, g in grupos.items()}
    sobrepostos = [(a, b) for a, b in combinations(ALGORITMOS, 2)
                   if sobrepoe(resumos[a], resumos[b])]
    return grupos, resumos, sobrepostos


decisao = pd.DataFrame([
    {"formato": f,
     "pares com ICs sobrepostos": ", ".join(
         "{} x {}".format(SIGLA[a], SIGLA[b]) for a, b in decide(f)[2]) or "nenhum",
     "desafio extra (ANOVA + Tukey)": "sim" if decide(f)[2] else "nao"}
    for f in FORMATOS
])
display(decisao)

,formato,pares com ICs sobrepostos,desafio extra (ANOVA + Tukey)
0,QR-QI-ME,nenhum,nao
1,QR-ME-QI,nenhum,nao
2,QI-QR-ME,QR x QI,sim
3,QI-ME-QR,QR x QI,sim
4,ME-QR-QI,QR x QI,sim
5,ME-QI-QR,nenhum,nao


## 2. Funções de análise

In [3]:
def analisa(formato):
    grupos, resumos, sobrepostos = decide(formato)
    cabecalho = "Formato " + " > ".join(formato.split("-"))
    print(cabecalho)
    print("-" * len(cabecalho))
    if not sobrepostos:
        print("Os ICs dos tres algoritmos nao se sobrepoem. A conclusao sai da comparacao "
              "dos ICs; a ANOVA nao e necessaria neste formato.")
        return

    print("ICs sobrepostos: " + ", ".join("{} x {}".format(SIGLA[a], SIGLA[b])
                                         for a, b in sobrepostos))
    print("Desafio extra acionado: ANOVA de um fator e Tukey HSD.")
    print()

    arrays = [grupos[a] for a in ALGORITMOS]

    print("Premissas")
    linhas = []
    for a in ALGORITMOS:
        W, p = stats.shapiro(grupos[a])
        linhas.append({"algoritmo": SIGLA[a], "Shapiro-Wilk W": W, "p-valor": p,
                       "normal (p > 0,05)": "sim" if p > ALFA else "nao"})
    display(pd.DataFrame(linhas).round(4))
    W_lev, p_lev = stats.levene(*arrays)
    print("Levene: estatistica = {:.4f}, p = {:.4g} -> variancias {}".format(
        W_lev, p_lev, "homogeneas" if p_lev > ALFA else "NAO homogeneas"))
    print()

    F, p_anova = stats.f_oneway(*arrays)
    k = len(arrays)
    N = sum(len(g) for g in arrays)
    media_geral = np.concatenate(arrays).mean()
    sq_entre = sum(len(g) * (g.mean() - media_geral) ** 2 for g in arrays)
    sq_dentro = sum(((g - g.mean()) ** 2).sum() for g in arrays)
    tabela = pd.DataFrame([
        {"fonte": "entre grupos", "SQ": sq_entre, "GL": k - 1,
         "QM": sq_entre / (k - 1), "F": F, "p-valor": p_anova},
        {"fonte": "dentro (erro)", "SQ": sq_dentro, "GL": N - k,
         "QM": sq_dentro / (N - k), "F": np.nan, "p-valor": np.nan},
        {"fonte": "total", "SQ": sq_entre + sq_dentro, "GL": N - 1,
         "QM": np.nan, "F": np.nan, "p-valor": np.nan},
    ])
    print("ANOVA de um fator (H0: as tres medias sao iguais)")
    display(tabela.round(4))
    eta2 = sq_entre / (sq_entre + sq_dentro)
    print("eta^2 = {:.3f} (fracao da variacao explicada pelo algoritmo)".format(eta2))
    print()

    res = stats.tukey_hsd(*arrays)
    ci = res.confidence_interval(confidence_level=CONFIANCA)
    linhas = []
    for i, j in combinations(range(k), 2):
        linhas.append({
            "par": "{} - {}".format(SIGLA[ALGORITMOS[i]], SIGLA[ALGORITMOS[j]]),
            "diferenca (ms)": res.statistic[i, j],
            "p-valor ajustado": res.pvalue[i, j],
            "IC 95% da diferenca (ms)": "[{:.2f}; {:.2f}]".format(ci.low[i, j], ci.high[i, j]),
            "diferente (p < 0,05)": "sim" if res.pvalue[i, j] < ALFA else "nao",
        })
    print("Tukey HSD (pares)")
    display(pd.DataFrame(linhas).round(4))
    print()

    if p_anova < ALFA:
        print("ANOVA significativa (p = {:.3g}): ha diferenca entre algum par de "
              "algoritmos. Os pares com 'sim' no Tukey sao os que diferem.".format(p_anova))
    else:
        print("ANOVA nao significativa (p = {:.3g}): nao ha evidencia de diferenca "
              "entre os tres algoritmos neste formato.".format(p_anova))

## 3. Análise por formato

### 3.1 Formato QR > QI > ME

In [4]:
analisa("QR-QI-ME")

Formato QR > QI > ME
--------------------
Os ICs dos tres algoritmos nao se sobrepoem. A conclusao sai da comparacao dos ICs; a ANOVA nao e necessaria neste formato.


### 3.2 Formato QR > ME > QI

In [5]:
analisa("QR-ME-QI")

Formato QR > ME > QI
--------------------
Os ICs dos tres algoritmos nao se sobrepoem. A conclusao sai da comparacao dos ICs; a ANOVA nao e necessaria neste formato.


### 3.3 Formato QI > QR > ME

In [6]:
analisa("QI-QR-ME")

Formato QI > QR > ME
--------------------
ICs sobrepostos: QR x QI
Desafio extra acionado: ANOVA de um fator e Tukey HSD.

Premissas


,algoritmo,Shapiro-Wilk W,p-valor,"normal (p > 0,05)"
0,QR,0.8752,0.0022,nao
1,QI,0.7806,0.0000,nao
2,ME,0.9324,0.0570,sim


Levene: estatistica = 0.6573, p = 0.5208 -> variancias homogeneas

ANOVA de um fator (H0: as tres medias sao iguais)


,fonte,SQ,GL,QM,F,p-valor
0,entre grupos,103539.1170,2,51769.5585,624.8204,0.0
1,dentro (erro),7208.3940,87,82.8551,NaN,NaN
2,total,110747.5109,89,NaN,NaN,NaN


eta^2 = 0.935 (fracao da variacao explicada pelo algoritmo)



Tukey HSD (pares)


,par,diferenca (ms),p-valor ajustado,IC 95% da diferenca (ms),"diferente (p < 0,05)"
0,QR - QI,-3.4976,0.3015,[-9.10; 2.11],nao
1,QR - ME,-73.6361,0.0000,[-79.24; -68.03],sim
2,QI - ME,-70.1385,0.0000,[-75.74; -64.53],sim



ANOVA significativa (p = 2.44e-52): ha diferenca entre algum par de algoritmos. Os pares com 'sim' no Tukey sao os que diferem.


### 3.4 Formato QI > ME > QR

In [7]:
analisa("QI-ME-QR")

Formato QI > ME > QR
--------------------
ICs sobrepostos: QR x QI
Desafio extra acionado: ANOVA de um fator e Tukey HSD.

Premissas


,algoritmo,Shapiro-Wilk W,p-valor,"normal (p > 0,05)"
0,QR,0.5712,0.0000,nao
1,QI,0.9164,0.0099,nao
2,ME,0.9658,0.3217,sim


Levene: estatistica = 0.0368, p = 0.9639 -> variancias homogeneas

ANOVA de um fator (H0: as tres medias sao iguais)


,fonte,SQ,GL,QM,F,p-valor
0,entre grupos,115229.6921,2,57614.8460,450.9099,0.0
1,dentro (erro),13416.3357,105,127.7746,NaN,NaN
2,total,128646.0278,107,NaN,NaN,NaN


eta^2 = 0.896 (fracao da variacao explicada pelo algoritmo)



Tukey HSD (pares)


,par,diferenca (ms),p-valor ajustado,IC 95% da diferenca (ms),"diferente (p < 0,05)"
0,QR - QI,-6.3130,0.051,[-12.65; 0.02],nao
1,QR - ME,-72.2314,0.000,[-78.57; -65.90],sim
2,QI - ME,-65.9184,0.000,[-72.25; -59.58],sim



ANOVA significativa (p = 2.87e-52): ha diferenca entre algum par de algoritmos. Os pares com 'sim' no Tukey sao os que diferem.


### 3.5 Formato ME > QR > QI

In [8]:
analisa("ME-QR-QI")

Formato ME > QR > QI
--------------------
ICs sobrepostos: QR x QI
Desafio extra acionado: ANOVA de um fator e Tukey HSD.

Premissas


,algoritmo,Shapiro-Wilk W,p-valor,"normal (p > 0,05)"
0,QR,0.7912,0.0000,nao
1,QI,0.6855,0.0000,nao
2,ME,0.8189,0.0001,nao


Levene: estatistica = 2.7187, p = 0.07157 -> variancias homogeneas

ANOVA de um fator (H0: as tres medias sao iguais)


,fonte,SQ,GL,QM,F,p-valor
0,entre grupos,107708.0003,2,53854.0001,426.7295,0.0
1,dentro (erro),10979.5500,87,126.2017,NaN,NaN
2,total,118687.5503,89,NaN,NaN,NaN


eta^2 = 0.907 (fracao da variacao explicada pelo algoritmo)



Tukey HSD (pares)


,par,diferenca (ms),p-valor ajustado,IC 95% da diferenca (ms),"diferente (p < 0,05)"
0,QR - QI,-3.9204,0.3709,[-10.84; 3.00],nao
1,QR - ME,-75.2669,0.0000,[-82.18; -68.35],sim
2,QI - ME,-71.3465,0.0000,[-78.26; -64.43],sim



ANOVA significativa (p = 1.07e-45): ha diferenca entre algum par de algoritmos. Os pares com 'sim' no Tukey sao os que diferem.


### 3.6 Formato ME > QI > QR

In [9]:
analisa("ME-QI-QR")

Formato ME > QI > QR
--------------------
Os ICs dos tres algoritmos nao se sobrepoem. A conclusao sai da comparacao dos ICs; a ANOVA nao e necessaria neste formato.
